# RAG Document Q&A Chatbot

**Stack:** PyTorch · HuggingFace Transformers · LangChain LCEL · FAISS · Pydantic · Gradio  
**LLM:** TinyLlama-1.1B-Chat (2048 token context, Apache 2.0)  
**Embedding:** all-MiniLM-L6-v2 (384-dim, pretrained, frozen)  

> ⚡ Runtime → Change runtime type → **T4 GPU** before running


## Cell 1 — Install dependencies


In [ ]:
# Run once, restart runtime if Colab asks you to

!pip install -q langchain langchain-community langchain-huggingface
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers faiss-cpu
!pip install -q PyMuPDF
!pip install -q transformers accelerate
!pip install -q gradio


## Cell 2 — Imports


In [ ]:

import fitz          # PyMuPDF - reads PDFs
import re
import os
import torch
import numpy as np
import requests
import warnings

warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer
import faiss

from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Imports done")
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Cell 3 — Load PDF (download from URL or upload from local machine)


In [ ]:
# Set USE_UPLOAD = False to auto-download Apple 10-K from url
# Set USE_UPLOAD = True  to upload your own PDF file

USE_UPLOAD = False   # change this to True if you want to upload your own PDF


def download_pdf(url, save_path="document.pdf"):
    """Download a PDF from a public URL."""
    print(f"Downloading from: {url}")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with open(save_path, "wb") as f:
        f.write(resp.content)
    print(f"Saved: {save_path} ({len(resp.content) // 1024} KB)")
    return save_path


def upload_pdf():
    """Upload a PDF from local machine (Colab only)."""
    from google.colab import files
    print("Select your PDF file in the picker below...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file selected.")
    filename = list(uploaded.keys())[0]
    save_path = f"/content/{filename}"
    with open(save_path, "wb") as f:
        f.write(uploaded[filename])
    print(f"Uploaded: {filename} ({len(uploaded[filename]) // 1024} KB)")
    return save_path


if not USE_UPLOAD:
    # Apple 10-K FY2025 - from SEC EDGAR
    PDF_URL = (
        "https://s2.q4cdn.com/470004039/files/doc_financials/2025/ar/_10-K-2025-As-Filed.pdf"
    )
    pdf_path = download_pdf(PDF_URL)
else:
    pdf_path = upload_pdf()

print(f"PDF ready: {pdf_path}")


## Cell 4 — Extract raw text from the PDF


In [ ]:

def extract_text_from_pdf(pdf_path):
    """Read every page and return all text as one string."""
    doc = fitz.open(pdf_path)
    pages = [doc[i].get_text("text") for i in range(len(doc))]
    doc.close()
    full_text = "\n".join(pages)
    print(f"Pages extracted: {len(pages)}")
    print(f"Total characters: {len(full_text):,}")
    print("\nSample (first 400 chars):")
    print(full_text[:400])
    return full_text


raw_text = extract_text_from_pdf(pdf_path)


## Cell 5 — Clean the extracted text


In [ ]:

def clean_text(text):
    """Remove noise from raw PDF text."""
    before = len(text)

    text = re.sub(r'\n{3,}', '\n\n', text)                    # collapse blank lines
    text = re.sub(r'[ \t]{2,}', ' ', text)                    # collapse spaces
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)               # strip non-ASCII
    text = re.sub(r'Apple Inc\.\s*\|.*?Form 10-K.*?\|\s*\d+', '', text, flags=re.IGNORECASE)  # page footers
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)          # lone page numbers
    text = re.sub(r'Table of Contents', '', text, flags=re.IGNORECASE)
    text = text.strip()

    print(f"Characters: {before:,} -> {len(text):,}")
    print("\nSample after cleaning:")
    print(text[:400])
    return text


cleaned_text = clean_text(raw_text)


## Cell 6 — Split text into overlapping chunks


In [ ]:
# chunk_size=800 captures full financial sentences
# chunk_overlap=100 prevents losing context at chunk boundaries

def chunk_text(text, chunk_size=800, chunk_overlap=100):
    """Split document into overlapping text passages."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_text(text)
    print(f"Total chunks: {len(chunks)}")
    print(f"\nChunk 0 sample:\n{chunks[0]}")
    print(f"\nChunk 1 sample:\n{chunks[1]}")
    return chunks


chunks = chunk_text(cleaned_text)


## Cell 7 — Load pretrained sentence embedding model (Transfer Learning)


In [ ]:
# all-MiniLM-L6-v2: pretrained on 1B+ sentence pairs, outputs 384-dim vectors
# We use it frozen - no fine-tuning needed

def load_embedding_model(model_name="all-MiniLM-L6-v2"):
    """Load pretrained sentence transformer for text -> vector conversion."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)
    print(f"Embedding model loaded: {model_name}")
    print(f"Device: {device} | Output dim: 384")
    return model


embed_model = load_embedding_model()


## Cell 8 — Encode all chunks and store in FAISS vector index


In [ ]:
# FAISS (Meta) does fast similarity search - this is our knowledge base

def build_vector_store(chunks, embed_model):
    """Encode all text chunks and build a FAISS index for similarity search."""
    print(f"Encoding {len(chunks)} chunks...")

    embeddings = embed_model.encode(
        chunks,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    print(f"Embeddings shape: {embeddings.shape}")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)          # L2 distance, exact search
    index.add(embeddings.astype(np.float32))

    print(f"FAISS index ready - {index.ntotal} vectors stored")
    return index, chunks


faiss_index, all_chunks = build_vector_store(chunks, embed_model)


## Cell 9 — Retriever: find top-k most relevant chunks for a query


In [ ]:

def retrieve_chunks(query, embed_model, faiss_index, chunks, top_k=5):
    """Return the top-k most semantically similar chunks for a given query."""
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = faiss_index.search(query_vec, top_k)
    return [chunks[i] for i in indices[0]]


# quick test
print("--- Retriever test ---")
test_q = "What was Apple's total revenue in 2025?"
results = retrieve_chunks(test_q, embed_model, faiss_index, all_chunks)
print(f"Query: {test_q}\n")
print("Top chunk:")
print(results[0])


## Cell 10 — Load TinyLlama-1.1B-Chat (Transfer Learning)


In [ ]:

def load_llm(model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
    """Load TinyLlama tokenizer + model. Returns tokenizer, model, device."""
    print(f"Loading {model_name}...")
    print("This downloads ~2.2 GB on first run, ~1 min on GPU")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto"   # auto-assigns layers to GPU/CPU based on available VRAM
    )
    model.eval()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Loaded: {params_m:.0f}M params | device: {device}")
    return tokenizer, model, device


tokenizer, model, device = load_llm()


## Cell 11 — LangChain imports for building the LCEL chain


In [ ]:
!pip install -q langchain pydantic

from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

print("LangChain imports done")


## Cell 12 — Structured output schema using Pydantic


In [ ]:

class RAGResponse(BaseModel):
    """Structured output for every RAG answer."""
    answer: str = Field(description="Direct answer extracted from the document.")
    confidence: str = Field(description="High / Medium / Low based on answer quality.")
    source_hint: str = Field(description="Which part of the document the answer came from.")


print("RAGResponse schema defined - fields: answer | confidence | source_hint")


## Cell 13 — Structured prompt template using LangChain PromptTemplate


In [ ]:

def build_prompt():
    """Build the LangChain prompt template for TinyLlama."""
    template = """<|system|>
You are a strict financial analyst. Your only job is to extract the exact answer from the context.
Rules:
1. Answer ONLY using facts explicitly stated in the context below.
2. If the exact answer is not in the context, respond with ONLY: not found
3. Do NOT summarize. Do NOT explain. Give the direct fact only.
4. Keep your answer under 2 sentences.</s>
<|user|>
Context:
{context}

Question: {question}</s>
<|assistant|>"""

    prompt = PromptTemplate(input_variables=["context", "question"], template=template)
    print("Prompt template ready")
    print(f"Input variables: {prompt.input_variables}")
    return prompt


structured_prompt = build_prompt()


## Cell 14 — Output parser: map raw LLM text -> RAGResponse Pydantic object


In [ ]:

def parse_structured_output(llm_output):
    """Convert raw TinyLlama string into a typed RAGResponse."""
    print(f"\n  [DEBUG] Raw TinyLlama output: {repr(llm_output)}")

    answer = llm_output.strip()

    not_found = ["not found", "not mentioned", "not provided",
                 "not available", "not stated", "cannot find", "i don't know"]
    contradictory = ["which is not found in the document",
                     "which is not in the document",
                     "but this is not found"]

    if not answer:
        return RAGResponse(answer="Not found in document", confidence="Low",
                          source_hint="Apple 10-K 2025 — retrieved via FAISS")

    if any(p in answer.lower() for p in contradictory):
        return RAGResponse(answer="Not found in document", confidence="Low",
                          source_hint="Apple 10-K 2025 — retrieved via FAISS")

    if any(p in answer.lower() for p in not_found):
        confidence = "Low"
    elif len(answer.split()) <= 40:
        confidence = "High"
    else:
        confidence = "Medium"

    return RAGResponse(answer=answer, confidence=confidence,
                      source_hint="Apple 10-K 2025 — retrieved via FAISS")


# quick smoke test
_t = "Apple's net sales were $416,161 million in fiscal 2025."
_r = parse_structured_output(_t)
print(f"\nParser test:")
print(f"  answer     : {_r.answer}")
print(f"  confidence : {_r.confidence}")
print(f"  source     : {_r.source_hint}")


## Cell 15 — Build the TinyLlama generate function returning closure


In [ ]:

def build_generate_fn(tokenizer, model, device):
    """Returns a closure that runs one TinyLlama inference pass."""

    def generate_fn(prompt_text):
        # LangChain passes StringPromptValue, not plain str - extract .text
        prompt_str = prompt_text.text if hasattr(prompt_text, "text") else str(prompt_text)

        inputs = tokenizer(prompt_str, return_tensors="pt",
                          max_length=2048, truncation=True)
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        prompt_len = input_ids.shape[1]

        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.3,    # low temp = more factual answers
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # slice off the input prompt - we only want the new tokens
        answer_ids = output_ids[0][prompt_len:]
        return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

    return generate_fn


generate_fn = build_generate_fn(tokenizer, model, device)

# smoke test
print("--- generate_fn smoke test ---")
_test = "<|system|>\nYou are helpful.</s>\n<|user|>\nSay hello in one word.</s>\n<|assistant|>"
print(f"Output: {repr(generate_fn(_test))}")


## Cell 15B — Assemble the LangChain LCEL chain


In [ ]:

def build_rag_chain(generate_fn, prompt_template):
    """Assemble the 3-step LCEL chain."""
    chain = (
        prompt_template
        | RunnableLambda(generate_fn)
        | RunnableLambda(parse_structured_output)
    )
    print("LCEL chain ready:")
    print("  PromptTemplate | RunnableLambda(generate_fn) | RunnableLambda(parse_structured_output)")
    return chain


rag_chain = build_rag_chain(generate_fn, structured_prompt)


## Cell 16 — ask() function with conversation memory


In [ ]:

def format_history(chat_history):
    """Format past Q&A turns as a readable string for the prompt."""
    if not chat_history:
        return ""
    lines = ["Previous conversation:"]
    for q, a in chat_history:
        lines.append(f"Human: {q}")
        lines.append(f"Assistant: {a}")
    return "\n".join(lines) + "\n\n"


def ask(question, embed_model, faiss_index, chunks, chain,
        chat_history=None, top_k=5):
    """
    Full RAG pipeline with memory.
    Retrieves relevant chunks, prepends history, invokes the chain.
    Returns a typed RAGResponse object.
    """
    if chat_history is None:
        chat_history = []

    # retrieve top-k chunks, pass top 3 to LLM (fits in 2048 token window)
    relevant = retrieve_chunks(question, embed_model, faiss_index, chunks, top_k=top_k)
    context = "\n\n".join(relevant[:3])

    # prepend conversation history to current question
    question_with_history = format_history(chat_history) + question

    response = chain.invoke({"context": context, "question": question_with_history})
    return response


# end-to-end test - two turns to verify memory works
print("--- End-to-end test ---")
q1 = "What was Apple's total net sales in fiscal year 2025?"
r1 = ask(q1, embed_model, faiss_index, all_chunks, rag_chain, chat_history=[])
print(f"\nTurn 1: {q1}")
print(f"Answer: {r1.answer} | Confidence: {r1.confidence}")

q2 = "How does that compare to 2024?"
r2 = ask(q2, embed_model, faiss_index, all_chunks, rag_chain,
         chat_history=[(q1, r1.answer)])
print(f"\nTurn 2: {q2}  <- follow-up (memory test)")
print(f"Answer: {r2.answer} | Confidence: {r2.confidence}")


## Cell 17 — Terminal chatbot with conversation memory


In [ ]:

def run_chatbot(embed_model, faiss_index, chunks, chain):
    """Interactive Q&A loop with conversation memory."""
    print("\n" + "=" * 55)
    print("  RAG Document Q&A Chatbot")
    print("  Commands: 'history' | 'clear' | 'quit'")
    print("=" * 55)
    print("\nTry asking:")
    print("  Q1: What was Apple's total revenue in 2025?")
    print("  Q2: How does that compare to 2024?  <- uses memory")

    chat_history = []

    while True:
        print()
        turn = len(chat_history) + 1
        question = input(f"Turn {turn}: ").strip()

        if question.lower() in ("quit", "exit", "q"):
            print(f"Session ended. Total turns: {len(chat_history)}")
            break

        if question.lower() == "clear":
            chat_history = []
            print("Memory cleared.")
            continue

        if question.lower() == "history":
            if not chat_history:
                print("No history yet.")
            else:
                for i, (q, a) in enumerate(chat_history, 1):
                    print(f"  [{i}] Q: {q}")
                    print(f"  [{i}] A: {a}")
            continue

        if not question:
            continue

        # show retrieved chunks
        print("\nRetrieving context...")
        top_chunks = retrieve_chunks(question, embed_model, faiss_index, chunks, top_k=5)
        for i, c in enumerate(top_chunks[:3]):
            print(f"  Chunk {i+1}: {c[:150]}...")

        if chat_history:
            print(f"\n[Memory: {len(chat_history)} previous turn(s) passed to LLM]")

        print("\nGenerating answer...")
        resp = ask(question, embed_model, faiss_index, chunks, chain,
                   chat_history=chat_history, top_k=5)

        print(f"\n{'─' * 50}")
        print(f"  Answer     : {resp.answer}")
        print(f"  Confidence : {resp.confidence}")
        print(f"  Source     : {resp.source_hint}")
        print(f"{'─' * 50}")

        chat_history.append((question, resp.answer))


run_chatbot(embed_model, faiss_index, all_chunks, rag_chain)


## Cell 18 — Gradio web UI with PDF upload + conversation memory


In [ ]:

!pip install -q gradio

import gradio as gr

# active document state - starts with Apple 10-K from Cell 8
_active_index  = faiss_index
_active_chunks = all_chunks
_active_doc    = "Apple 10-K 2025 (default)"


def process_uploaded_pdf(pdf_file):
    """Rebuild the FAISS index when user uploads a new PDF."""
    global _active_index, _active_chunks, _active_doc

    if pdf_file is None:
        return "No file uploaded. Using default: Apple 10-K 2025."

    try:
        path = pdf_file.name
        raw        = extract_text_from_pdf(path)
        clean      = clean_text(raw)
        new_chunks = chunk_text(clean)
        new_index, new_chunks = build_vector_store(new_chunks, embed_model)

        _active_index  = new_index
        _active_chunks = new_chunks
        _active_doc    = path.split("/")[-1]

        return f"Loaded: {_active_doc}\nChunks: {len(new_chunks)}\nReady!"
    except Exception as e:
        return f"Error: {e}\nStill using previous document."


def chat_fn(question, gradio_history):
    """Called by Gradio on every user message. Returns bot reply string."""
    if not question.strip():
        return "Please type a question."

    # Gradio passes history as [[user, bot], ...] - convert to our (q,a) tuples
    history = [(h[0], h[1]) for h in gradio_history if h[1] is not None]

    resp = ask(question, embed_model, _active_index, _active_chunks,
               rag_chain, chat_history=history, top_k=5)

    return f"{resp.answer}\n\nConfidence: {resp.confidence}\nSource: {resp.source_hint}"


with gr.Blocks(title="RAG Document Q&A") as demo:

    gr.Markdown(
        "# RAG Document Q&A Chatbot\n"
        "Upload any PDF and ask questions about it.\n\n"
        "**Stack:** FAISS · TinyLlama-1.1B-Chat · LangChain LCEL · Pydantic"
    )

    gr.Markdown("## Step 1 — Upload a PDF (optional)")
    with gr.Row():
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
        doc_status = gr.Textbox(
            label="Status",
            value="Default: Apple 10-K 2025. Upload a PDF to switch documents.",
            lines=3,
            interactive=False
        )
    pdf_input.change(fn=process_uploaded_pdf, inputs=pdf_input, outputs=doc_status)

    gr.Markdown("---")
    gr.Markdown("## Step 2 — Ask Questions")

    gr.ChatInterface(
        fn=chat_fn,
        examples=[
            "What was the total net sales in fiscal year 2025?",
            "How does that compare to 2024?",
            "What are the main products and services?",
            "What risks are mentioned about AI or tariffs?",
            "What is the net income?",
        ],
        title=""
    )

demo.launch(share=True)
